B24. Verbosity bias судьи: любит ли рубрика длинное.....


In [1]:
import sys, os, re

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/evaluation_robustness')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
import statsmodels.api as sm
A = load_arms()
print()
d0 = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
e = pd.to_numeric(d0['judge_earned'], errors='coerce')
pn = pd.to_numeric(d0['judge_penalties'], errors='coerce')
mx = pd.to_numeric(d0['judge_max'], errors='coerce')
sc = pd.to_numeric(d0['Judge_score'], errors='coerce')
print('проверка формулы Judge_score = (earned − penalties) / max:')
print('   максимальное расхождение = %.2e' % ((e - pn) / mx - sc).abs().max())
print('   доля кейсов со штрафами = %.3f, средний штраф = %.3f' % ((pn > 0).mean(), pn.mean()))


плеч загружено: 18
   baseline gpt, baseline qwen, baseline deepseekv4, baseline deepseekv4pro, baseline gemini, baseline glm, baseline kimi, USC@3 gpt, USC@5 gpt, USC@7 gpt, homog gpt, homog qwen, homog gemini, homog deepseekv4, P-weak, P-mixed, P-strong, P-base



проверка формулы Judge_score = (earned − penalties) / max:
   максимальное расхождение = 5.00e-05
   доля кейсов со штрафами = 0.052, средний штраф = 0.214


In [2]:
def load_retest(arm, design='noroles'):
    tpl = RETEST.get(arm)
    if tpl is None:
        return None
    parts = []
    for r in range(1, 6):
        d = rd(tpl % (design, r))
        if d is None:
            continue
        col = 'winner_protocol' if 'winner_protocol' in d.columns else None
        if col is None:
            continue
        parts.append(pd.DataFrame({'arm': arm, 'run': r, 'case': d.index, 'score': pd.to_numeric(d['Judge_score'], errors='coerce').to_numpy(), 'earned': pd.to_numeric(d.get('judge_earned'), errors='coerce').to_numpy(), 'penalties': pd.to_numeric(d.get('judge_penalties'), errors='coerce').to_numpy(), 'maxpts': pd.to_numeric(d.get('judge_max'), errors='coerce').to_numpy(), 'words': d[col].map(words).to_numpy()}))
    return pd.concat(parts, ignore_index=True) if parts else None
RT = pd.concat([x for x in (load_retest(a) for a in RETEST) if x is not None], ignore_index=True).dropna(subset=['score', 'words'])
RT = RT[RT['words'] > 0].copy()
RT['lw'] = np.log(RT['words'])
print('наблюдений (система × прогон × кейс):', len(RT))
show(RT.groupby('arm').agg(прогонов=('run', 'nunique'), кейсов=('case', 'nunique'), слов_медиана=('words', 'median'), разброс_слов_внутри_кейса=('words', lambda s: np.nan)), 'Материал естественного эксперимента')
wi = RT.groupby(['arm', 'case'])['words'].agg(['min', 'max', 'std']).reset_index()
wi['размах'] = wi['max'] - wi['min']
show(wi.groupby('arm')[['размах', 'std']].mean(), 'Разброс длины между прогонами ВНУТРИ одного кейса')
print()
print('Именно этот разброс мы используем как «случайное» варьирование длины.')


наблюдений (система × прогон × кейс): 1200

Материал естественного эксперимента
------------------------------------------------------------
                     прогонов  кейсов  слов_медиана  разброс_слов_внутри_кейса
arm                                                                           
P-base                      5      60      466.0000                        NaN
baseline deepseekv4         5      60      314.0000                        NaN
baseline gpt                5      60      289.0000                        NaN
baseline qwen               5      60      268.0000                        NaN

Разброс длины между прогонами ВНУТРИ одного кейса
------------------------------------------------------------
                      размах     std
arm                                 
P-base              137.1333 55.6862
baseline deepseekv4  93.6500 38.2563
baseline gpt         65.1500 26.7769
baseline qwen        50.6500 20.7906

Именно этот разброс мы используем как «случайное» 

In [3]:
RT['cell'] = RT['arm'] + '|' + RT['case'].astype(str)
D = pd.get_dummies(RT['cell'], prefix='fe', drop_first=True).astype(float)
X = sm.add_constant(pd.concat([RT[['lw']], D], axis=1))
g = RT['cell'].astype('category').cat.codes
m = sm.OLS(RT['score'].astype(float), X).fit(cov_type='cluster', cov_kwds={'groups': g})
print('ВНУТРИ пары (система, кейс): эффект удвоения длины на Judge_score')
print('   коэффициент при log(слов) = %+.5f   SE = %.5f   p = %s' % (m.params['lw'], m.bse['lw'], fmt_p(m.pvalues['lw'])))
print('   эффект удвоения длины     = %+.5f балла' % (m.params['lw'] * np.log(2)))
print('   R² = %.4f' % m.rsquared)
print()
print('Для сравнения, между системами (из C13): коэффициент около +0.19,')
print('то есть удвоение длины там связано с приростом около +0.13 балла.')


ВНУТРИ пары (система, кейс): эффект удвоения длины на Judge_score
   коэффициент при log(слов) = +0.22125   SE = 0.03040   p = 3.4e-13
   эффект удвоения длины     = +0.15336 балла
   R² = 0.8065

Для сравнения, между системами (из C13): коэффициент около +0.19,
то есть удвоение длины там связано с приростом около +0.13 балла.


In [4]:
rows = []
for dep, lab in [('score', 'Judge_score'), ('earned', 'заработано баллов'), ('penalties', 'штрафы'), ('maxpts', 'максимум рубрики')]:
    y = pd.to_numeric(RT[dep], errors='coerce')
    ok = y.notna()
    mm = sm.OLS(y[ok].astype(float), sm.add_constant(pd.concat([RT.loc[ok, ['lw']], D.loc[ok]], axis=1))).fit(cov_type='cluster', cov_kwds={'groups': g[ok]})
    rows.append({'зависимая': lab, 'коэф при log(слов)': mm.params['lw'], 'SE': mm.bse['lw'], 'p': mm.pvalues['lw']})
show(pd.DataFrame(rows).set_index('зависимая'), 'Через что действует длина (внутри кейса и системы)')
print()
print('ПРО СТРОКУ «максимум рубрики». Она задумывалась как контроль-негатив,')
print('но в этой спецификации ВЫРОЖДЕНА: максимум постоянен внутри кейса,')
print('а фиксированный эффект кейса поглощает его полностью. Поэтому')
print('коэффициент равен ровно нулю с нулевой стандартной ошибкой, и p там')
print('бессмысленно. Строку следует читать как проверку того, что')
print('фиксированные эффекты работают, а не как содержательный тест.')
print()
print('СОДЕРЖАТЕЛЬНЫЙ ВЫВОД: длина действует через «заработано баллов»')
print('(+14.7 балла на e-кратное удлинение), а не через штрафы (p = 0.21).')



Через что действует длина (внутри кейса и системы)
------------------------------------------------------------
                   коэф при log(слов)     SE      p
зависимая                                          
Judge_score                    0.2213 0.0304 0.0000
заработано баллов             14.6936 1.8838 0.0000
штрафы                        -0.6512 0.5248 0.2146
максимум рубрики               0.0000 0.0000 0.0000

ПРО СТРОКУ «максимум рубрики». Она задумывалась как контроль-негатив,
но в этой спецификации ВЫРОЖДЕНА: максимум постоянен внутри кейса,
а фиксированный эффект кейса поглощает его полностью. Поэтому
коэффициент равен ровно нулю с нулевой стандартной ошибкой, и p там
бессмысленно. Строку следует читать как проверку того, что
фиксированные эффекты работают, а не как содержательный тест.

СОДЕРЖАТЕЛЬНЫЙ ВЫВОД: длина действует через «заработано баллов»
(+14.7 балла на e-кратное удлинение), а не через штрафы (p = 0.21).


In [5]:
rows = []
for arm, tpl in JUDGE41.items():
    d_main = A.get(arm)
    d_alt = rd(tpl % 'noroles')
    if d_main is None or d_alt is None:
        continue
    col = 'winner_protocol' if 'winner_protocol' in d_main.columns else 'baseline_protocol_anonymous'
    w = d_main[col].map(words)
    for lab, dd in [('основной судья', d_main), ('gpt-4.1', d_alt)]:
        s = pd.to_numeric(dd['Judge_score'], errors='coerce')
        ix = s.dropna().index.intersection(w.dropna().index)
        if len(ix) < 30:
            continue
        X2 = sm.add_constant(pd.DataFrame({'lw': np.log(w.loc[ix])}))
        mm = sm.OLS(s.loc[ix].astype(float), X2).fit(cov_type='HC3')
        rows.append({'плечо': arm, 'судья': lab, 'n': len(ix), 'коэф при log(слов)': mm.params['lw'], 'SE': mm.bse['lw'], 'p': mm.pvalues['lw'], 'R²': mm.rsquared})
JS = pd.DataFrame(rows)
show(JS.set_index(['плечо', 'судья']), 'Наклон по длине у двух судей')
if len(JS):
    piv = JS.pivot(index='плечо', columns='судья', values='коэф при log(слов)')
    if piv.shape[1] == 2:
        piv['разница'] = piv.iloc[:, 0] - piv.iloc[:, 1]
        show(piv, 'Совпадают ли наклоны')
        print()
        print('Близкие наклоны = длина связана с содержанием, а не с капризом')
        print('одного судьи. Разные наклоны = у судей разное отношение к длине,')
        print('и абсолютные баллы нельзя сравнивать между судьями.')



Наклон по длине у двух судей
------------------------------------------------------------
                                      n  коэф при log(слов)     SE      p     R²
плечо               судья                                                       
baseline gpt        основной судья  229              0.1587 0.0466 0.0007 0.0375
                    gpt-4.1          60              0.2440 0.0823 0.0030 0.1225
baseline qwen       основной судья  229              0.2088 0.0662 0.0016 0.0468
                    gpt-4.1          60              0.2327 0.1858 0.2103 0.0568
baseline deepseekv4 основной судья  228              0.1792 0.0553 0.0012 0.0517
                    gpt-4.1          59              0.1460 0.1021 0.1526 0.0387

Совпадают ли наклоны
------------------------------------------------------------
судья                gpt-4.1  основной судья  разница
плечо                                                
baseline deepseekv4   0.1460          0.1792  -0.0332
baseline gpt    

In [6]:
ITEMS, POS = load_items(verbose=False)
rows = []
for arm in ITEMS:
    d = A.get(arm)
    if d is None:
        continue
    col = 'winner_protocol' if 'winner_protocol' in d.columns else 'baseline_protocol_anonymous'
    if col not in d.columns:
        continue
    w = d[col].map(words)
    for k in POS:
        if k[0] not in w.index or not np.isfinite(w.get(k[0], np.nan)):
            continue
        rows.append({'arm': arm, 'item': '%s#%d' % (k[0], k[1]), 'met': int(ITEMS[arm][k]['met']), 'words': float(w[k[0]])})
IT = pd.DataFrame(rows)
IT = IT[IT['words'] > 0].copy()
IT['lw'] = np.log(IT['words'])
print('наблюдений (пункт × плечо):', len(IT), ' пунктов:', IT['item'].nunique(), ' плеч:', IT['arm'].nunique())
samp = IT[IT['item'].isin(pd.Series(IT['item'].unique()).sample(700, random_state=0))]
Xi = pd.get_dummies(samp['item'], prefix='i', drop_first=True).astype(float)
Xa = pd.get_dummies(samp['arm'], prefix='a', drop_first=True).astype(float)
mod1 = sm.Logit(samp['met'].astype(float), sm.add_constant(pd.concat([samp[['lw']], Xi], axis=1))).fit(disp=0, method='lbfgs', maxiter=300)
mod2 = sm.Logit(samp['met'].astype(float), sm.add_constant(pd.concat([samp[['lw']], Xi, Xa], axis=1))).fit(disp=0, method='lbfgs', maxiter=300)
show(pd.DataFrame({'только эффект пункта': [mod1.params['lw'], mod1.pvalues['lw'], np.exp(mod1.params['lw'])], '+ эффект плеча': [mod2.params['lw'], mod2.pvalues['lw'], np.exp(mod2.params['lw'])]}, index=['коэф при log(слов)', 'p', 'OR на e-кратное удлинение']), 'Вероятность «пункт выполнен» ~ длина протокола (выборка 700 пунктов)')
print()
print('ЧИТАЕТСЯ ТАК. Первый столбец: длина сильно повышает шанс засчитать')
print('пункт — но туда попадает и то, что сильные системы пишут длиннее.')
print('Второй столбец добавляет фиксированный эффект ПЛЕЧА, то есть')
print('сравнивает разные кейсы внутри одной системы. Если коэффициент там')
print('остаётся большим, длина работает и внутри одной системы.')


наблюдений (пункт × плечо): 40482  пунктов: 2249  плеч: 18



Вероятность «пункт выполнен» ~ длина протокола (выборка 700 пунктов)
--------------------------------------------------------------------
                           только эффект пункта  + эффект плеча
коэф при log(слов)                       3.3373          2.4773
p                                        0.0000          0.0000
OR на e-кратное удлинение               28.1434         11.9091

ЧИТАЕТСЯ ТАК. Первый столбец: длина сильно повышает шанс засчитать
пункт — но туда попадает и то, что сильные системы пишут длиннее.
Второй столбец добавляет фиксированный эффект ПЛЕЧА, то есть
сравнивает разные кейсы внутри одной системы. Если коэффициент там
остаётся большим, длина работает и внутри одной системы.


D:\PythonUser\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [7]:
w_panel = A['P-base']['winner_protocol'].map(words).median()
w_base = A['baseline gpt']['winner_protocol'].map(words).median() if 'winner_protocol' in A['baseline gpt'].columns else A['baseline gpt']['baseline_protocol_anonymous'].map(words).median()
gap = np.log(w_panel) - np.log(w_base)
beta_within = m.params['lw']
attrib = beta_within * gap
delta_total = float((js(A, 'P-base') - js(A, 'baseline gpt')).mean())
print('медиана длины P-base       : %.0f слов' % w_panel)
print('медиана длины baseline gpt : %.0f слов' % w_base)
print('разрыв в log-шкале         : %.3f' % gap)
print()
print('коэффициент «внутри системы и кейса» (шаг 1): %+.5f' % beta_within)
print('приписываемая длине часть эффекта          : %+.5f' % attrib)
print('фактический эффект панели                  : %+.5f' % delta_total)
print('доля, объяснимая чистым эффектом длины     : %.1f%%' % (100 * attrib / delta_total))
print()
print('=' * 72)
print('КАК ЭТО ЧИТАТЬ — И ЧЕГО ЗДЕСЬ НЕЛЬЗЯ УТВЕРЖДАТЬ')
print('=' * 72)
print('Полученная доля — это ВЕРХНЯЯ ГРАНИЦА того, что могло бы объясняться')
print('длиной, а НЕ доказательство смещения судьи. Причина в том, что внутри')
print('одной системы и одного кейса длина тоже не случайна: прогон получился')
print('длиннее потому, что модели было что сказать. Регрессия не может')
print('отличить «написал больше и потому заслуженно получил больше» от')
print('«получил больше просто за объём».')
print()
print('Что мы всё-таки узнали:')
print('  1) связь длины с баллом сохраняется при фиксированных системе и кейсе,')
print('     то есть она не сводится к «сильные модели пишут длиннее»;')
print('  2) она идёт через засчитанные пункты, а не через штрафы;')
print('  3) оба независимых судьи показывают положительный наклон схожей')
print('     величины — значит это не каприз одного оценщика.')
print()
print('Что НЕ узнали и чем это закрывается: нужен каузальный тест —')
print('заставить систему изложить ТО ЖЕ САМОЕ короче или длиннее и сравнить')
print('баллы. Одно дополнительное плечо на 229 кейсов. До него утверждение')
print('«судья вознаграждает многословие» остаётся гипотезой, а величина 43%%')
print('из C13 и величина выше — двумя границами одного и того же интервала.')


медиана длины P-base       : 472 слов
медиана длины baseline gpt : 285 слов
разрыв в log-шкале         : 0.504

коэффициент «внутри системы и кейса» (шаг 1): +0.22125
приписываемая длине часть эффекта          : +0.11162
фактический эффект панели                  : +0.16414
доля, объяснимая чистым эффектом длины     : 68.0%

КАК ЭТО ЧИТАТЬ — И ЧЕГО ЗДЕСЬ НЕЛЬЗЯ УТВЕРЖДАТЬ
Полученная доля — это ВЕРХНЯЯ ГРАНИЦА того, что могло бы объясняться
длиной, а НЕ доказательство смещения судьи. Причина в том, что внутри
одной системы и одного кейса длина тоже не случайна: прогон получился
длиннее потому, что модели было что сказать. Регрессия не может
отличить «написал больше и потому заслуженно получил больше» от
«получил больше просто за объём».

Что мы всё-таки узнали:
  1) связь длины с баллом сохраняется при фиксированных системе и кейсе,
     то есть она не сводится к «сильные модели пишут длиннее»;
  2) она идёт через засчитанные пункты, а не через штрафы;
  3) оба независимых судьи показыв

In [8]:
rows = []
for arm in ['P-base', 'baseline gpt', 'homog gpt', 'USC@7 gpt', 'P-strong']:
    d = A.get(arm)
    if d is None or 'judge_penalties' not in d.columns:
        continue
    col = 'winner_protocol' if 'winner_protocol' in d.columns else 'baseline_protocol_anonymous'
    w = d[col].map(words)
    p_ = pd.to_numeric(d['judge_penalties'], errors='coerce')
    ix = w.dropna().index.intersection(p_.dropna().index)
    r, pv = stats.spearmanr(w.loc[ix], p_.loc[ix])
    rows.append({'плечо': arm, 'n': len(ix), 'доля со штрафами': float((p_.loc[ix] > 0).mean()), 'ср. штраф': float(p_.loc[ix].mean()), 'rho (длина, штрафы)': r, 'p': pv})
show(pd.DataFrame(rows).set_index('плечо'), 'Штрафы и длина протокола')



Штрафы и длина протокола
------------------------------------------------------------
                n  доля со штрафами  ср. штраф  rho (длина, штрафы)      p
плечо                                                                     
P-base        229            0.0524     0.2140               0.0993 0.1340
baseline gpt  229            0.2358     1.2620               0.0075 0.9100
homog gpt     229            0.0961     0.4716               0.0178 0.7889
USC@7 gpt     229            0.1965     1.0873              -0.0107 0.8725
P-strong      229            0.0699     0.2926              -0.1139 0.0854


In [9]:
save(RT, 'judge_verbosity_bias_retest_lengths.csv')
save(JS, 'judge_verbosity_bias_grader_swap.csv')


сохранено: out/judge_verbosity_bias_retest_lengths.csv  (1200 строк)
сохранено: out/judge_verbosity_bias_grader_swap.csv  (6 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\evaluation_robustness\\results\\judge_verbosity_bias_grader_swap.csv'